# Trabajo Práctico 3: Redes Neuronales

El objetivo de este trabajo es desarrollar y evaluar modelos basados en redes neuronales,
incorporando técnicas de ablación para entender el impacto de diversas modificaciones en el
proceso de entrenamiento y en la capacidad de generalización del modelo.

## 0. Import modules

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# This enviromental variables change the cpu/threads usage of numpy
import os
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["VECLIB_MAXIMUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import zipfile # for treating zip
sys.path.append('../')

# import modules of project
from src.utils import *
from src.models import *

# random state for every lib and mod using np
np.random.seed(42)

# config for future plots and pd visualization
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 1. Análisis y Preprocesamiento de Datos

### a) Examinar el dataset y visualizar al menos 3 imágenes
### b) Dividir el conjunto de datos en tres subconjuntos: Train, Validation y Test.
### c) Normalizar dividiendo todos los valores por 255, de modo que el máximo sea 1.
---

In [ ]:
# First thing first, let's open the datasets
with zipfile.ZipFile('../data/datasets.zip', 'r') as z:
    with z.open('I302 ML 2025 2do semestre TP3_ datasets/X_images.npy') as f:
        X_images = np.load(f)
    with z.open('I302 ML 2025 2do semestre TP3_ datasets/y_images.npy') as f:
        y_images = np.load(f)
    with z.open('I302 ML 2025 2do semestre TP3_ datasets/X_COMP.npy') as f:
        X_COMP = np.load(f)


In [ ]:
# lets inspect the images
idxs = np.random.choice(len(X_images), size=5, replace=False)
plt.figure(figsize=(15, 3))
plt.title("Sample images from EMNIST")
plt.axis('off')
for i, idx in enumerate(idxs):
    plt.subplot(1, 5, i + 1)
    
    img = X_images[idx].reshape(28, 28)
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')

plt.show()

In [ ]:
print(f"Shape of X_images: {X_images.shape}\nShape of y_images: {y_images.shape}")
num_clases = len(np.unique(y_images))
print(f"Number of classes: {num_clases}")
X_images = X_images / 255.0
print(f"Pixel min_value: {X_images.min()}\nPixel max_value: {X_images.max()}")

Parece que hay 47 clases de números (y letras dado que trabajamos con Extended MNIST). Esto es algo importante ya que las redes usarán Binary Cross Entropy (BCE) y de alguna forma tiene que estar encodeado numéricamente cada label (más allá del número de clase con el que ya se cuenta). La idea es hacer one hot encoding. Si la complejidad computacional se ve alterada fuertemente por el aumento de la dimensión del dataset, entonces vamos a evaluar otro tipo de encoding.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(
    X_images,
    y_images,
    train_ratio=0.70,
    val_ratio=0.15
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
show_label_distribution(y_images, "Original")
show_label_distribution(y_train, "Train")
show_label_distribution(y_val, "Validation")
show_label_distribution(y_test, "Test")

Lo que nos muestran los gráficos, la distribución de las categorías del target se mantiene, por lo cual garantizamos una estratificación representativa para el entrenamiento, validación y testeo.

In [ ]:
X_train_flat = flatten_images(X_train)
X_val_flat = flatten_images(X_val)
X_test_flat = flatten_images(X_test)

assert X_train_flat.shape[1] == 784
assert X_val_flat.shape[1] == 784
assert X_test_flat.shape[1] == 784

In [ ]:
num_classes = len(np.unique(y_images))

y_train_one_hot = one_hot_encode(y_train, num_classes)
y_val_one_hot = one_hot_encode(y_val, num_classes)
y_test_one_hot = one_hot_encode(y_test, num_classes)

Finalmente, flateamos y encodeamos el target para poder usar en la CE

## 2. Implementación y Entrenamiento de una Red Neuronal Básica

### a) Implementar una red neuronal Multi-layer perceptron (MLP) con $L$ capas ocultas, cada una con $M(l)$ nodos, utilizando ReLU en las capas ocultas y activación softmax en la capa de salida.
### b) Implementar un algoritmo para entrenar dicha red, mediante backpropagation y gradiente descendente estándar, utilizando como función de costo la cross-entropy. NOTA: El algoritmo backpropagation debe adaptarse al caso de clasificación multiclase con función de activación softmax en la salida y función de costo cross-entropy.
### c) Entrenar una red neuronal con $2$ capas ocultas, con $128$ y $64$ nodos respectivamente, y graficar la evolución de la función de costo (cross-entropy) sobre los conjuntos de entrenamiento y validación a lo largo de las épocas. Llamaremos a este modelo M0.
### d) Reportar las siguientes métricas de performance, sobre los conjuntos de entrenamiento y validación, para el modelo base entrenado: Accuracy, Cross-Entropy, Matriz de Confusión, F1-Score Macro
---

In [ ]:
M0 = MLPClassifier(
    layer_sizes=[784, 128, 64, num_classes],
    learning_rate=0.1
)

result_M0, history_M0 = train_and_evaluate_model(
    model_name="M0",
    change_description="Base model",
    model=M0,
    X_train=X_train_flat,
    y_train=y_train_one_hot,
    X_val=X_val_flat,
    y_val=y_val_one_hot,
    plot=True,
    train_config={
        "epochs": 50,
        "batch_size": len(X_train_flat),
        "optimizer": "gd",
        "scheduler": None,
        "initial_lr": 0.1,
        "l2_lambda": 0.0,
        "early_stopping": False,
        "verbose": True
    }
)

## 3. Implementación y Entrenamiento de una Red Neuronal Avanzada
### a) Implementar las siguientes mejoras al algoritmo de entrenamiento, y para cada una reportar el efecto observado sobre el tiempo de entrenamiento y la performance del modelo resultante.
- Rate scheduling lineal (con saturación) y exponencial.
- Mini-batch stochastic gradient descent.
- Optimizador Adam.
- Regularización (L2, Early Stopping)
- Opcional: Batch normalization.
- Opcional: Label smoothing.
- Opcional: Gradient clipping.
---

### A1 — Mini-batch SGD

In [ ]:
a1 = MLPClassifier(
    layer_sizes=[784, 128, 64, num_classes],
    learning_rate=0.1
)

result_a1, history_a1 = train_and_evaluate_model(
    model_name="A1",
    change_description="Mini-batch SGD",
    model=a1,
    X_train=X_train_flat,
    y_train=y_train_one_hot,
    X_val=X_val_flat,
    y_val=y_val_one_hot,
    train_config={
        "epochs": 50,
        "batch_size": 128,
        "optimizer": "gd",
        "scheduler": None,
        "initial_lr": 0.1,
        "l2_lambda": 0.0,
        "early_stopping": False,
        "verbose": True
    }
)

### A2 — Adam

In [ ]:
a2 = MLPClassifier(
    layer_sizes=[784, 128, 64, num_classes],
    learning_rate=0.1
)

result_a2, history_a2 = train_and_evaluate_model(
    model_name="A2",
    change_description="Adam",
    model=a2,
    X_train=X_train_flat,
    y_train=y_train_one_hot,
    X_val=X_val_flat,
    y_val=y_val_one_hot,
    train_config={
        "epochs": 50,
        "batch_size": len(X_train_flat),
        "optimizer": "adam",
        "scheduler": None,
        "initial_lr": 0.1,
        "l2_lambda": 0.0,
        "early_stopping": False,
        "verbose": True
    }
)

### A3 — L2 + early stopping

In [ ]:
a3 = MLPClassifier(
    layer_sizes=[784, 128, 64, num_classes],
    learning_rate=0.1
)

result_a3, history_a3 = train_and_evaluate_model(
    model_name="A3",
    change_description="L2",
    model=a3,
    X_train=X_train_flat,
    y_train=y_train_one_hot,
    X_val=X_val_flat,
    y_val=y_val_one_hot,
    train_config={
        "epochs": 50,
        "batch_size": len(X_train_flat),
        "optimizer": "gd",
        "scheduler": None,
        "initial_lr": 0.1,
        "l2_lambda": 1e-4,
        "early_stopping": True,
        "verbose": True
    }
)

### A4 — Exponential LR Scheduler

In [ ]:
a4 = MLPClassifier(
    layer_sizes=[784, 128, 64, num_classes],
    learning_rate=0.1
)

result_a4, history_a4 = train_and_evaluate_model(
    model_name="A4",
    change_description="Exponential LR Scheduler",
    model=a4,
    X_train=X_train_flat,
    y_train=y_train_one_hot,
    X_val=X_val_flat,
    y_val=y_val_one_hot,
    train_config={
        "epochs": 50,
        "batch_size": len(X_train_flat),
        "optimizer": "gd",
        "scheduler": "exponential",
        "initial_lr": 0.1,
        "final_lr": 1e-5,
        "decay_rate": 0.97,
        "l2_lambda": 0.0,
        "early_stopping": False,
        "verbose": True
    }
)

### A5 — Batch Normalization

In [ ]:
a5 = MLPClassifier(
    layer_sizes=[784, 128, 64, num_classes],
    learning_rate=0.1,
    use_batch_norm=True
)

result_a5, history_a5 = train_and_evaluate_model(
    model_name="A5",
    change_description="Batch Normalization",
    model=a5,
    X_train=X_train_flat,
    y_train=y_train_one_hot,
    X_val=X_val_flat,
    y_val=y_val_one_hot,
    train_config={
        "epochs": 50,
        "batch_size": len(X_train_flat),
        "optimizer": "gd",
        "scheduler": "None",
        "initial_lr": 0.1,
        "l2_lambda": 0.0,
        "early_stopping": False,
        "verbose": True
    }
)

### A6 — Label Smoothing

In [ ]:
a6 = MLPClassifier(
    layer_sizes=[784, 128, 64, num_classes],
    learning_rate=0.1
)

result_a6, history_a6 = train_and_evaluate_model(
    model_name="A6",
    change_description="Label Smoothing",
    model=a6,
    X_train=X_train_flat,
    y_train=y_train_one_hot,
    X_val=X_val_flat,
    y_val=y_val_one_hot,
    train_config={
        "epochs": 50,
        "batch_size": len(X_train_flat),
        "optimizer": "gd",
        "scheduler": "None",
        "initial_lr": 0.1,
        "l2_lambda": 0.0,
        "early_stopping": False,
        "label_smoothing": 0.05,
        "verbose": True
    }
)

### A7 — Gradient Clipping

In [ ]:
a7 = MLPClassifier(
    layer_sizes=[784, 128, 64, num_classes],
    learning_rate=0.1
)

result_a7, history_a7 = train_and_evaluate_model(
    model_name="A7",
    change_description="Gradient Clipping",
    model=a7,
    X_train=X_train_flat,
    y_train=y_train_one_hot,
    X_val=X_val_flat,
    y_val=y_val_one_hot,
    train_config={
        "epochs": 50,
        "batch_size": len(X_train_flat),
        "optimizer": "gd",
        "scheduler": "None",
        "initial_lr": 0.1,
        "l2_lambda": 0.0,
        "early_stopping": False,
        "gradient_clip_norm": 5.0,
        "verbose": True
    }
)

In [ ]:
ablation_table = pd.DataFrame([
    result_M0,
    result_a1,
    result_a2,
    result_a3,
    result_a4,
    result_a5,
    result_a6,
    result_a7
])

ablation_table

### b) Explorar cambios en la arquitectura de la red (es decir, la cantidad de capas ocultas y unidades ocultas por capa), y en los hiperparámetros (cada uno de los ítems en la lista anterior tiene parámetros que se pueden variar), y determinar la configuración que funcione mejor (menor error de validación). Llamaremos a este modelo M1.
---

M1 se define como la configuración con menor cross-entropy de validación. Pero sobre múltiples combinaciones de hiperparámetros lo estadísticamente correcto sería hacer cross validation. Por ello, se realizó una búsqueda de hiperparámetros mediante K-Fold estratificado sobre el conjunto de desarrollo, construido a partir de train y validation. Los folds se generaron preservando la distribución de clases. La selección de M1 se hizo minimizando la cross-entropy promedio de validación entre folds. El conjunto de test se mantuvo aislado para la comparación final de modelos.

## 4. Desarrollo de una Red Neuronal con PyTorch
### a) Utilizando PyTorch, entrenar una red neuronal con la arquitectura y los hiperparámetros hallados en el ejercicio anterior. Llamaremos a este modelo M2. Comparar la performance del modelo M1 y M2 para validar que los comportamientos de los modelos son parecidos.
### b) Utilizando PyTorch, explorar cambios en la cantidad de capas ocultas y unidades ocultas por capa, y determinar la configuración que funcione mejor. Además de la arquitectura, se deberá experimentar con funciones de activación más modernas disponibles en torch.nn, tales como LeakyReLU, SiLU, Swish o GELU y estrategias de regularizacion adicionales como Dropout (ademas de las ya mencionadas previamente). Llamaremos a este modelo M3.
### c) Comparar la performance sobre el conjunto de test de los siguientes cuatro modelos:
1) El modelo base de implementación propia (M0).
2) La mejor arquitectura obtenida con la implementación propia (M1).
3) Modelo en PyTorch, usando la misma arquitectura e hiperparámetros que en la implementación propia (M2).
4) La mejor arquitectura obtenida en PyTorch (M3).
### d) Con los modelos ya entrenados, perturbar los datos de test con distintos niveles de ruido y evaluar cuan robusta es la performance de cada modelo frente a estos datos perturbados en comparacion a los resultados obtenidos de datos sin perturbar.